# 1. Data collection

**Source.** CORDIS, the European Commission's public database of EU-funded research projects. We use the official bulk exports released monthly by the Publications Office of the EU on the [EU Open Data Portal](https://data.europa.eu).

**Dataset.** [Horizon Europe (2021–2027)](https://data.europa.eu/data/datasets/cordis-eu-research-projects-under-horizon-europe-2021-2027), distribution *HORIZON Projects*, in two formats. The two archives are downloaded by this notebook into `data/raw/`:

| file in `data/raw/` | format | table used |
|---|---|---|
| `horizon_europe.zip` | CSV | participations, `organization.csv` |
| `horizon_europe_json.zip` | JSON | projects, rebuilt as `project_json.csv` |

**Tables.** The participations have one row for each organisation in each project, with its country, activity type and role in the consortium. The projects have one row per project, with its start date.

**Why two formats.** The participations are read from the CSV export. The projects are rebuilt from the JSON export, because in the CSV export the project table contains unescaped quotes inside the abstracts, and a small share of its rows cannot be parsed.

**Output.** The two tables are saved, compressed, in `data/snapshot/`, which is kept in the repository. They are the input of `02_build_network.ipynb`, so the analyses always run on the same data. The archives in `data/raw/` are not in the repository (see `.gitignore`) because of their size: running this notebook downloads them again, in the version available at that time.

**Snapshot date.** 28 September 2026 (dump version reported on the dataset page). Dumps are refreshed monthly, so a later download may yield slightly different counts: the analyses are not affected, because they run on the copy kept in `data/snapshot/`, which is never replaced.

## Imports and configuration

In [1]:
import gzip
import hashlib
import json
import shutil
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

# Folder of the downloaded zip archives
RAW_DIR = Path("../data/raw")
# Folder of the snapshot kept in the repository: the two tables used in the project, compressed
SNAPSHOT_DIR = Path("../data/snapshot")

# Archives to download: for each one, the name of the file saved in data/raw/ and the web address it comes from
ARCHIVES = {
    "horizon_europe.zip": "https://cordis.europa.eu/data/cordis-HORIZONprojects-csv.zip",
    "horizon_europe_json.zip": "https://cordis.europa.eu/data/cordis-HORIZONprojects-json.zip",
}

## Downloading the archives

The archives are downloaded from CORDIS the first time the notebook is run. Later runs reuse the archives already in `data/raw/`. If a download fails, for instance because an address is no longer valid, the notebook reports it: the archive can then be downloaded from the dataset page linked above and saved in `data/raw/` with the file name of the table.

In [2]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

for filename, url in ARCHIVES.items():
    path = RAW_DIR / filename

    # An archive already present is kept, so that the analysis stays on the same snapshot
    if path.exists():
        print(f"{filename}: already in {RAW_DIR}, not downloaded again")
        continue

    # Download the archive from CORDIS and save it in data/raw/
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as response, open(path, "wb") as file:
            shutil.copyfileobj(response, file)
        print(f"{filename}: downloaded")

    # If the download fails, remove the incomplete file and say what to do
    except Exception as error:
        path.unlink(missing_ok=True)
        print(f"{filename}: download failed ({error})")
        print(f"  Download it from the dataset page and save it as {path}")

horizon_europe.zip: already in ..\data\raw, not downloaded again
horizon_europe_json.zip: already in ..\data\raw, not downloaded again


For each archive we compute its size and its SHA-256 checksum, a code that identifies the content of a file. An archive with a different checksum is a different version of the data.

In [3]:
def sha256(path):
    """Return the SHA-256 checksum of a file, read in blocks so that a large archive does not fill the memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Size and checksum of each archive in data/raw/
checksums = {}
print(f"{'archive':26}{'size (MB)':>10}   SHA-256")
for filename in ARCHIVES:
    path = RAW_DIR / filename
    if path.exists():
        checksums[filename] = sha256(path)
        print(f"{filename:26}{path.stat().st_size / 1e6:>10.1f}   {checksums[filename]}")
    else:
        print(f"{filename:26}{'missing':>10}")

archive                    size (MB)   SHA-256
horizon_europe.zip              36.9   1496a16ee79649d573a53697781b2fd950c18a5469c6a9f545f35dd5397771a7
horizon_europe_json.zip        141.1   5327f9338d844f9e50ab70e65cd2205975dce18351cec150eb11f3743b908acf


## Building the snapshot

The two tables are read from the archives and saved, compressed, in `data/snapshot/`, with the checksums of the archives they come from. The snapshot is built only the first time. In later runs it is never replaced: the notebook only says whether the archives in `data/raw/` are the same version or a different one. The following notebooks read the snapshot, so their results do not depend on when the data are downloaded.

In [4]:
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# File with the checksums of the archives the snapshot was built from: it exists only when the snapshot is complete
checksum_file = SNAPSHOT_DIR / "archives_sha256.json"
build_snapshot = not checksum_file.exists()

if build_snapshot:
    print(f"No snapshot in {SNAPSHOT_DIR} yet: it is built from the archives in {RAW_DIR}")
else:
    # The snapshot exists: compare the archives in data/raw/ with those it was built from
    if len(checksums) < len(ARCHIVES):
        print(f"The archives are not all in {RAW_DIR}, so they cannot be compared with the snapshot.")
    elif checksums == json.loads(checksum_file.read_text()):
        print(f"The archives in {RAW_DIR} are those of the snapshot.")
    else:
        print(f"The archives in {RAW_DIR} are a different version from the snapshot.")
    print(f"The snapshot in {SNAPSHOT_DIR} is kept: the following notebooks read it, and their results do not change.")

The archives in ..\data\raw are those of the snapshot.
The snapshot in ..\data\snapshot is kept: the following notebooks read it, and their results do not change.


### Participations (CSV export)

The participation table, `organization.csv`, is copied from the CSV archive to the snapshot without extracting the archive. The other tables of the archive are not used.

In [5]:
if build_snapshot:
    with zipfile.ZipFile(RAW_DIR / "horizon_europe.zip") as archive:

        # Name of the participation table inside the archive
        member = next(name for name in archive.namelist() if name.endswith("organization.csv"))

        # Copy the table to the snapshot, compressed
        with archive.open(member) as source, gzip.open(SNAPSHOT_DIR / "organization.csv.gz", "wb") as target:
            shutil.copyfileobj(source, target)

    print("organization.csv.gz saved")

### Projects (JSON export)

In the JSON export every project is a separate file. We read each file directly from the archive and keep two fields, found at the top level of the file: the identifier and the start date. The result is saved in the snapshot as `project_json.csv.gz`.

In [6]:
if build_snapshot:

    # Fields to keep from the top level of each project JSON file
    FIELDS = ["id", "startDate"]

    rows = []
    with zipfile.ZipFile(RAW_DIR / "horizon_europe_json.zip") as archive:
        for filename in archive.namelist():

            # Skip anything that is not a project file
            if not filename.endswith(".json"):
                continue
            with archive.open(filename) as f:
                project = json.loads(f.read().decode("utf-8", errors="replace"))

            # Row of the project: value of each field
            rows.append({field: project.get(field) for field in FIELDS})

    # Save the table in the snapshot, compressed
    pd.DataFrame(rows).to_csv(SNAPSHOT_DIR / "project_json.csv.gz", index=False, compression="gzip")
    print(f"project_json.csv.gz saved ({len(rows)} projects)")

    # The checksums of the archives are saved last, so that an interrupted run does not leave a snapshot that looks complete
    checksum_file.write_text(json.dumps(checksums, indent=2))

## Sanity check

We load the two tables of the snapshot, which are the input of `02_build_network.ipynb`, and check that they are read correctly: number of rows, number of distinct organisations (the future nodes) and available columns. CORDIS CSV files use `;` as separator.

In [7]:
# Load the two tables of the snapshot into pandas DataFrames
projects = pd.read_csv(SNAPSHOT_DIR / "project_json.csv.gz", low_memory=False)
participations = pd.read_csv(SNAPSHOT_DIR / "organization.csv.gz", sep=";", low_memory=False,
                             keep_default_na=False, na_values=[""])

# Basic counts
print("projects:               ", len(projects))
print("participations:         ", len(participations))
print("distinct organisations: ", participations["organisationID"].nunique())

projects:                23613
participations:          145909
distinct organisations:  35200


In [8]:
# Columns available in the two tables
print("project_json.csv columns:")
print(list(projects.columns), "\n")

print("organization.csv columns:")
print(list(participations.columns))

project_json.csv columns:
['id', 'startDate'] 

organization.csv columns:
['projectID', 'projectAcronym', 'organisationID', 'vatNumber', 'name', 'shortName', 'SME', 'activityType', 'street', 'postCode', 'city', 'country', 'nutsCode', 'geolocation', 'organizationURL', 'contactForm', 'contentUpdateDate', 'rcn', 'order', 'role', 'ecContribution', 'netEcContribution', 'totalCost', 'endOfParticipation', 'active']
